# Домашнее задание 1. Агент, среда, награда

**Вес в оценке:** базовое ДЗ, лёгкое. Цель — освоиться с интерфейсом Gymnasium и научиться
описывать задачи на языке RL.

Задание состоит из трёх частей:

1. **Практика (50 баллов)** — запуск эпизодов в Gymnasium и политики, написанные руками
2. **Теория (30 баллов)** — три задачи из жизни на языке RL
3. **Мини-эксперимент (20 баллов)** — исследование vs использование

Части задания с `assert` проверяются автоматически при запуске ячейки — если assert не
упал, эта часть засчитана. Текстовые ответы и графики проверяются вручную.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

rng = np.random.default_rng(0)

## Часть 1. Практика (50 баллов)

### 1.1 Запуск эпизодов (10 баллов)

Реализуйте функцию `run_episodes(env_id, policy_fn, n_episodes, seed)`, которая создаёт среду
`gym.make(env_id)`, прогоняет `n_episodes` эпизодов с политикой `policy_fn(obs) -> action`
и возвращает массив суммарных наград по эпизодам. Эпизод `ep` сбрасывайте через
`env.reset(seed=seed + ep)`, чтобы результаты были воспроизводимы. Эпизод заканчивается,
когда `terminated or truncated`.

In [ ]:
def run_episodes(env_id: str, policy_fn, n_episodes: int = 20, seed: int = 0) -> np.ndarray:
    # TODO: ваш код здесь
    raise NotImplementedError

In [ ]:
# Самопроверка: случайная политика на CartPole держит шест в среднем 15-35 шагов
def random_cartpole(obs):
    return int(rng.integers(2))

_G = run_episodes("CartPole-v1", random_cartpole, n_episodes=50, seed=0)
assert _G.shape == (50,), "функция должна возвращать массив длины n_episodes"
assert 10 < _G.mean() < 40, f"средняя награда случайной политики {_G.mean():.1f} выглядит подозрительно"
print(f"OK: run_episodes работает, случайная политика: {_G.mean():.1f} ± {_G.std():.1f}")

### 1.2 Ручная политика для CartPole (20 баллов)

Напишите политику `my_cartpole_policy(obs) -> {0, 1}`, которая держит шест **в среднем не меньше
400 шагов** по 30 эпизодам (максимум 500). Наблюдение: `[x, x_dot, theta, theta_dot]`.

Подсказки: эвристика из лекции (`theta_dot > 0`) даёт около 200; добавьте угол `theta`; чтобы тележка
не уезжала за край, можно учитывать `x` и `x_dot` с небольшими весами. Хорошая форма — взвешенная сумма
всех четырёх чисел с порогом. Веса подбирайте руками, это часть задания.

In [ ]:
def my_cartpole_policy(obs) -> int:
    x, x_dot, theta, theta_dot = obs
    # TODO: ваш код здесь
    raise NotImplementedError

In [ ]:
_G = run_episodes("CartPole-v1", my_cartpole_policy, n_episodes=30, seed=100)
print(f"ваша политика: {_G.mean():.1f} ± {_G.std():.1f}, минимум {_G.min():.0f}, максимум {_G.max():.0f}")
assert _G.mean() >= 400, "нужно в среднем не меньше 400 шагов"
print("OK: политика для CartPole засчитана")

### 1.3 Политики-таблицы для FrozenLake (20 баллов)

Таблица-политика — словарь `состояние -> действие` (действия: `0` влево, `1` вниз, `2` вправо, `3` вверх)
для **всех 16 клеток** карты `FrozenLake-v1` (карта и нумерация есть в семинаре). Нужны две таблицы:

* `frozen_policy_flat` (5 баллов): на гладком льду (`is_slippery=False`) агент доходит до цели в **100%** эпизодов.
* `frozen_policy_slippery` (15 баллов): на скользком льду (`is_slippery=True`) агент доходит до цели
  **не реже чем в 30%** эпизодов из 500. Маршрут «напрямую» даёт всего несколько процентов, так что
  придётся думать: куда агента может снести из каждой клетки, и как использовать стены (упереться
  в стену — безопасное действие, а снос иногда работает в нужную сторону). Одна и та же таблица
  для обеих версий не получится, и это нормально. Бонус +5 баллов, если доля успехов ≥ 60%
  (лучшая возможная — около 75%).
  Если не получается, опишите текстом, что вы пробовали и почему это не сработало, — часть баллов за это.

In [ ]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

frozen_policy_flat = {
    # TODO: заполните все 16 состояний, например 0: DOWN, ...
}

frozen_policy_slippery = {
    # TODO: заполните все 16 состояний
}


def frozen_success_rate(table: dict, slippery: bool, n_episodes: int = 500) -> float:
    env = gym.make("FrozenLake-v1", is_slippery=slippery)
    wins = 0
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        while True:
            obs, reward, terminated, truncated, _ = env.step(table[int(obs)])
            if terminated or truncated:
                wins += reward > 0
                break
    env.close()
    return wins / n_episodes

In [ ]:
assert len(frozen_policy_flat) == 16 and len(frozen_policy_slippery) == 16, "нужно задать действие для всех 16 клеток"
_flat = frozen_success_rate(frozen_policy_flat, slippery=False)
_slip = frozen_success_rate(frozen_policy_slippery, slippery=True)
print(f"гладкий лёд: {_flat:.1%}, скользкий лёд: {_slip:.1%}")
assert _flat == 1.0, "на гладком льду нужно доходить всегда"
assert _slip >= 0.3, "на скользком льду нужно доходить не реже чем в 30% эпизодов"
print("OK: политики для FrozenLake засчитаны" + (", бонус за скользкий лёд" if _slip >= 0.6 else ""))

## Часть 2. Теория: задачи из жизни на языке RL (30 баллов)

Для каждой из трёх задач ниже опишите, как её сформулировать как задачу обучения с подкреплением.
Отвечайте прямо в markdown-ячейке, по пунктам:

* **агент** — кто принимает решения;
* **среда** — что отвечает на действия и что в ней случайного;
* **состояние / наблюдение** — что агент видит на каждом шаге (и чего *не* видит);
* **действия** — из чего выбирает;
* **награда** — число за один шаг, и как оно связано с настоящей целью;
* **эпизод** — что считать началом и концом, или задача бесконечная;
* **где здесь дилемма exploration / exploitation** — что значит «попробовать новое» в этой задаче.

Задачи (10 баллов каждая):

1. **Торговля на бирже**: программа управляет портфелем из нескольких акций.
2. **Охлаждение дата-центра**: программа управляет насосами и вентиляцией, чтобы серверы не перегревались.
3. **На выбор**: рой дронов, который должен обыскать территорию, **или** любая задача из вашей жизни
   (обучение, спорт, игра, работа), которую вы сможете описать по тем же пунктам.

Бонус (+5 баллов): для одной из задач придумайте пример **reward hacking** — как агент может получить
много награды, не делая того, что вы на самом деле хотели, и как поправить награду.

### 2.1 Торговля на бирже

_Ваш ответ:_ TODO

### 2.2 Охлаждение дата-центра

_Ваш ответ:_ TODO

### 2.3 Задача на выбор

_Ваш ответ:_ TODO

### Бонус: reward hacking

_Ваш ответ:_ TODO

## Часть 3. Мини-эксперимент: исследовать или использовать (20 баллов)

Ниже — симуляция «трёх кафе» из лекции. Агент каждый день выбирает кафе: с вероятностью
`explore_prob` случайное, иначе лучшее по текущей средней оценке.

Задания:

1. (10 баллов) Постройте график: по оси x — `explore_prob` из `np.linspace(0, 1, 21)`, по оси y — среднее
   удовольствие за день, усреднённое по 200 запускам. Отметьте, при каком `explore_prob` результат максимален.
2. (5 баллов) Повторите для `n_days = 30` и `n_days = 2000`. Как меняется лучший `explore_prob` и почему?
3. (5 баллов) Ответьте текстом: почему чисто жадный агент (`explore_prob = 0`) проигрывает, хотя каждый день
   делает «лучший по его мнению» выбор? Что именно он делает неправильно?

In [ ]:
true_means = np.array([5.0, 6.0, 7.5])


def simulate(explore_prob: float, n_days: int = 200, rng=None) -> float:
    rng = rng or np.random.default_rng()
    estimate = np.zeros(3)
    visits = np.zeros(3)
    total = 0.0
    for day in range(n_days):
        if rng.random() < explore_prob or visits.min() == 0:
            k = int(rng.integers(3))
        else:
            k = int(np.argmax(estimate))
        reward = np.clip(true_means[k] + rng.normal(0, 2.0), 0, 10)
        visits[k] += 1
        estimate[k] += (reward - estimate[k]) / visits[k]
        total += reward
    return total / n_days


# TODO: ваш код здесь (графики для заданий 1 и 2)

_Ответ на вопрос 3:_ TODO

## Чек-лист перед сдачей

- [ ] `run_episodes` реализована и проходит self-check
- [ ] `my_cartpole_policy` держит шест в среднем ≥ 400 шагов
- [ ] `frozen_policy_flat` и `frozen_policy_slippery` заданы для всех 16 клеток и проходят проверки (или описано, что пробовали)
- [ ] Три задачи описаны по всем пунктам (агент, среда, состояние, действия, награда, эпизод, exploration)
- [ ] Построены графики для эксперимента с кафе, отвечен вопрос про жадного агента
- [ ] (бонус) придуман пример reward hacking и способ его починить